In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Dominick's: raw CSV -> bronze -> silver
# MAGIC
# MAGIC **Bronze** (`dominicks.bronze.*`): every movement and UPC-description CSV loaded as-is (all strings) into one Delta table each, plus `category`, `_source_file`, `_ingested_at`.
# MAGIC
# MAGIC **Silver** (`dominicks.silver.*`): typed, validated, joined to UPC descriptions (and the week map if configured). Rows that fail checks go to `silver.movement_rejects` with a reason instead of being dropped.
# MAGIC
# MAGIC **Tracking**: every step appends a row to `dominicks.meta.transform_log` (row counts, reject breakdown, run id). Table-to-table steps are captured in Unity Catalog lineage automatically.
# MAGIC
# MAGIC **Assumptions to verify** (standard Dominick's column conventions, not checked against your files):
# MAGIC - Movement columns: `store, upc, week, move, qty, price, sale, profit, ok`; UPC files have `upc, descrip`
# MAGIC - `ok = 1` means the row is usable; `qty` is the bundle size that `price` applies to; a non-blank `sale` code means a promotion
# MAGIC
# MAGIC Set `ONLY = ["oatmeal"]` below for a quick test run on one small category before loading everything.
# MAGIC **Note**: Silver isn't fully complete, I still will drop off some rows.

# COMMAND ----------

import os
import json
import uuid
from pathlib import Path

from pyspark.sql import functions as F

CATALOG = "dominicks"
RAW = f"/Volumes/{CATALOG}/raw"
MOVEMENT_DIR = f"{RAW}/movement"
UPC_DIR = f"{RAW}/upc_desc"

# Optional: path to the week map (parquet or csv) with columns `week` and one of date / date_min / week_date.
# Example: f"{RAW}/week_decode/week_map.parquet". Leave None to skip the week join.
WEEK_MAP_PATH = None

ONLY = None            # e.g. ["oatmeal"] to test with a single category
CLUSTER_SILVER = True  # liquid clustering on (category, store, week) after the silver build (v good)

BRONZE_MOVEMENT = f"{CATALOG}.bronze.movement"
BRONZE_UPC = f"{CATALOG}.bronze.upc_desc"
SILVER_MOVEMENT = f"{CATALOG}.silver.movement"
SILVER_REJECTS = f"{CATALOG}.silver.movement_rejects"
SILVER_UPC = f"{CATALOG}.silver.upc_desc"
LOG_TABLE = f"{CATALOG}.meta.transform_log"
RUN_ID = uuid.uuid4().hex[:8]
print("run id:", RUN_ID)

# COMMAND ----------

for schema in ["bronze", "silver", "meta"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
  run_id STRING, step STRING, source STRING, target STRING,
  rows_in BIGINT, rows_out BIGINT, detail STRING, logged_at TIMESTAMP
)""")


def log_step(step, source, target, rows_in, rows_out, detail=""):
    """Append one row describing a pipeline step to the transform log."""
    row = [(RUN_ID, step, source, target, rows_in, rows_out, detail)]
    schema = (
        "run_id string, step string, source string, target string, "
        "rows_in long, rows_out long, detail string"
    )
    (
        spark.createDataFrame(row, schema)
        .withColumn("logged_at", F.current_timestamp())
        .write.mode("append")
        .saveAsTable(LOG_TABLE)
    )

# COMMAND ----------

def csv_stems(folder):
    """{'analgesics': '/Volumes/.../analgesics.csv', ...}"""
    return {
        Path(n).stem: f"{folder}/{n}"
        for n in sorted(os.listdir(folder))
        if n.lower().endswith(".csv")
    }


movement_files = csv_stems(MOVEMENT_DIR)
upc_files = csv_stems(UPC_DIR)
if ONLY:
    movement_files = {k: v for k, v in movement_files.items() if k in ONLY}
    upc_files = {k: v for k, v in upc_files.items() if k in ONLY}

print(f"{len(movement_files)} movement files, {len(upc_files)} upc_desc files")
print("movement without upc_desc:", sorted(set(movement_files) - set(upc_files)))
print("upc_desc without movement:", sorted(set(upc_files) - set(movement_files)))

# COMMAND ----------
# MAGIC %md
# MAGIC ## Bronze: CSV -> Delta (all strings, nothing dropped)

# COMMAND ----------

def load_bronze(files, target, required, source):
    """First file overwrites the table (so reruns rebuild it), the rest append."""
    for i, (category, path) in enumerate(files.items()):
        raw = spark.read.option("header", True).option("inferSchema", False).csv(path)
        names = [c.strip().lower() for c in raw.columns]
        missing = [c for c in required if c not in names]
        if missing:
            raise ValueError(f"{path}: missing columns {missing}; found {names}")

        df = raw.select(
            *[F.col(f"`{c}`").alias(c.strip().lower()) for c in raw.columns],
            F.lit(category).alias("category"),
            F.col("_metadata.file_path").alias("_source_file"),
            F.current_timestamp().alias("_ingested_at"),
        )
        writer = df.write.format("delta")
        if i == 0:
            writer = writer.mode("overwrite").option("overwriteSchema", "true")
        else:
            writer = writer.mode("append").option("mergeSchema", "true")
        writer.saveAsTable(target)
        print(f"[{i + 1}/{len(files)}] {category} -> {target}")

    n = spark.table(target).count()  # Delta row counts come from metadata, so this is cheap
    log_step("bronze_load", f"{len(files)} CSV files in {source}", target, None, n)
    return n


n_move = load_bronze(
    movement_files, BRONZE_MOVEMENT,
    required=["store", "upc", "week", "move", "qty", "price", "sale", "profit", "ok"],
    source=MOVEMENT_DIR,
)
n_upc = load_bronze(upc_files, BRONZE_UPC, required=["upc", "descrip"], source=UPC_DIR)
print(f"bronze movement rows: {n_move:,}   bronze upc rows: {n_upc:,}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Silver: type, validate, enrich, join
# MAGIC Casts use `try_cast`, so a bad value becomes null (and is rejected with a reason) instead of failing the run.

# COMMAND ----------

MOVEMENT_TYPES = {
    "store": "int", "upc": "bigint", "week": "int", "move": "int",
    "qty": "int", "price": "double", "profit": "double", "ok": "int",
}


def type_movement(bronze):
    cols = [F.expr(f"try_cast(`{c}` as {t})").alias(c) for c, t in MOVEMENT_TYPES.items()]
    return bronze.select("category", *cols, F.col("sale"), F.col("_source_file"))


def add_reject_reason(df):
    """First matching rule wins; null reason means the row is clean."""
    return df.withColumn(
        "reject_reason",
        F.when(F.col("store").isNull() | F.col("upc").isNull() | F.col("week").isNull(), "null_key")
        .when(F.col("ok").isNull() | (F.col("ok") != 1), "ok_flag_not_1")
        .when(F.col("price").isNull() | (F.col("price") <= 0), "nonpositive_price")
        .when(F.col("qty").isNull() | (F.col("qty") <= 0), "nonpositive_qty")
        .when(F.col("move").isNull() | (F.col("move") < 0), "invalid_move"),
    )


def add_derived(clean):
    """Only call on rows that passed validation (qty > 0)."""
    return (
        clean.withColumn("unit_price", F.col("price") / F.col("qty"))
        .withColumn("revenue", F.col("unit_price") * F.col("move"))
        .withColumn("is_promo", F.col("sale").isNotNull() & (F.trim(F.col("sale")) != ""))
    )


def build_upc(bronze_upc):
    return (
        bronze_upc.withColumn("upc", F.expr("try_cast(upc as bigint)"))
        .where(F.col("upc").isNotNull())
        .drop("_source_file", "_ingested_at")
        .dropDuplicates(["category", "upc"])
    )


def load_week_map(path):
    if path is None:
        return None
    reader = spark.read.parquet if path.endswith(".parquet") else spark.read.option("header", True).csv
    wm = reader(path)
    names = [c.lower() for c in wm.columns]
    wm = wm.toDF(*names)
    date_col = next((c for c in ("date", "date_min", "week_date") if c in names), None)
    if "week" not in names or date_col is None:
        raise ValueError(f"{path}: need `week` and a date column; found {names}")
    return (
        wm.select(
            F.expr("try_cast(week as int)").alias("week"),
            F.expr(f"try_cast(`{date_col}` as date)").alias("week_date"),
        )
        .where(F.col("week").isNotNull())
        .dropDuplicates(["week"])
    )


def build_silver(clean, upc, week_map=None):
    out = clean.join(upc, ["category", "upc"], "left")
    if week_map is not None:
        out = out.join(week_map, "week", "left")
    return out

# COMMAND ----------

typed = type_movement(spark.table(BRONZE_MOVEMENT))
flagged = add_reject_reason(typed)

# rejects: kept for audit, never silently dropped
(
    flagged.where(F.col("reject_reason").isNotNull())
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(SILVER_REJECTS)
)
n_rej = spark.table(SILVER_REJECTS).count()
reasons = {
    r["reject_reason"]: r["count"]
    for r in spark.table(SILVER_REJECTS).groupBy("reject_reason").count().collect()
}
log_step("silver_validate", BRONZE_MOVEMENT, SILVER_REJECTS, n_move, n_rej, json.dumps(reasons))
print("rejected rows:", f"{n_rej:,}", reasons)

# UPC descriptions
(
    build_upc(spark.table(BRONZE_UPC))
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(SILVER_UPC)
)
log_step("silver_upc", BRONZE_UPC, SILVER_UPC, n_upc, spark.table(SILVER_UPC).count(), "dedupe on (category, upc)")

# movement: clean rows + derived columns + joins
clean = add_derived(flagged.where(F.col("reject_reason").isNull()).drop("reject_reason"))
week_map = load_week_map(WEEK_MAP_PATH)
silver = build_silver(clean, spark.table(SILVER_UPC), week_map)
(
    silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(SILVER_MOVEMENT)
)

n_silver = spark.table(SILVER_MOVEMENT).count()
n_nodesc = spark.table(SILVER_MOVEMENT).where(F.col("descrip").isNull()).count()
detail = {
    "rows_without_upc_description": n_nodesc,
    "week_map": WEEK_MAP_PATH,
    "derived": ["unit_price = price / qty", "revenue = unit_price * move", "is_promo = sale not blank"],
}
log_step("silver_build", f"{BRONZE_MOVEMENT} + {SILVER_UPC}", SILVER_MOVEMENT, n_move, n_silver, json.dumps(detail))
print(f"silver rows: {n_silver:,}   without UPC description: {n_nodesc:,}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Metadata for the semantic layer (comments and properties) and query speed

# COMMAND ----------

spark.sql(
    f"COMMENT ON TABLE {SILVER_MOVEMENT} IS "
    "'Weekly store-level scanner movement by UPC, typed and validated, joined to UPC descriptions'"
)
spark.sql(f"ALTER TABLE {SILVER_MOVEMENT} SET TBLPROPERTIES ('pipeline.run_id' = '{RUN_ID}')")

COLUMN_NOTES = {
    "unit_price": "Price per single unit: price divided by qty (verify against the Dominicks manual)",
    "revenue": "Dollar sales for the row: unit_price times move",
    "is_promo": "True when the sale code is not blank",
    "week_date": "Calendar date for the week number, from the week map",
    "move": "Units sold in the week",
}
existing = set(spark.table(SILVER_MOVEMENT).columns)
for col, text in COLUMN_NOTES.items():
    if col in existing:
        spark.sql(f"ALTER TABLE {SILVER_MOVEMENT} ALTER COLUMN {col} COMMENT '{text}'")

if CLUSTER_SILVER:
    try:
        spark.sql(f"ALTER TABLE {SILVER_MOVEMENT} CLUSTER BY (category, store, week)")
        spark.sql(f"OPTIMIZE {SILVER_MOVEMENT}")
        log_step("silver_cluster", SILVER_MOVEMENT, SILVER_MOVEMENT, None, None, "CLUSTER BY (category, store, week); OPTIMIZE")
    except Exception as e:  # clustering is a speed-up only, never fail the pipeline on it
        print("Clustering skipped:", str(e)[:300])

# COMMAND ----------
# MAGIC %md
# MAGIC ## Check the run

# COMMAND ----------

display(spark.table(LOG_TABLE).where(F.col("run_id") == RUN_ID).orderBy("logged_at"))

# COMMAND ----------

display(spark.table(SILVER_MOVEMENT).groupBy("category").count().orderBy("category"))

# COMMAND ----------

display(spark.table(SILVER_REJECTS).groupBy("category", "reject_reason").count().orderBy("category", "reject_reason"))

# COMMAND ----------

display(spark.sql(f"DESCRIBE HISTORY {SILVER_MOVEMENT}"))

# COMMAND ----------
# MAGIC %md
# MAGIC ## Where to see lineage
# MAGIC - Catalog -> `dominicks.silver.movement` -> **Lineage** tab: bronze movement, silver UPC descriptions, and the notebook that wrote it. Click a column for column-level lineage.
# MAGIC - Programmatic: `SELECT * FROM system.access.table_lineage WHERE target_table_full_name LIKE 'dominicks.%'`
# MAGIC - The raw CSV -> bronze step reads from a volume path, so it may not appear in the graph automatically. `bronze.movement._source_file` records the file for each row.